# Retail Banking Customer Analytics
**Dataset:** Bank Marketing (UCI / Kaggle) — Portuguese retail bank, 11,162 customer records  
**Business Question:** Which customer segments are most valuable, and what drives product uptake and churn risk?  
**Tools:** PostgreSQL · psycopg2 · pandas · matplotlib · seaborn

In [ ]:
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

conn = psycopg2.connect(
    host='localhost',
    dbname='bank_marketing',
    user='postgres',
    password='yuanlong',
    port=5432
)

def run_query(sql):
    return pd.read_sql_query(sql, conn)

sns.set_theme(style='whitegrid')
print('Connected successfully.')

---
## Section 1 — Customer Profiling

### Q1: Age Distribution by Job Type

In [ ]:
q1 = run_query("""
SELECT job, COUNT(*) AS total_customers,
    ROUND(AVG(age), 1) AS avg_age,
    MIN(age) AS min_age, MAX(age) AS max_age,
    PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY age) AS median_age
FROM bank GROUP BY job ORDER BY avg_age DESC;
""")
display(q1)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=q1.sort_values('avg_age'), x='avg_age', y='job', ax=ax, palette='Blues_r')
ax.set_title('Average Age by Job Segment')
ax.set_xlabel('Average Age')
plt.tight_layout()
plt.show()

**Business commentary:**  
Retired customers are the oldest segment (avg 65, median 64), while students are the youngest at 26. The bank's largest segment is management at 2,566 customers with an average age of 40 — the core mid-career demographic. The bulk of the customer base clusters between 37–42 years old, suggesting the bank should anchor its core product suite (mortgages, savings plans, investment products) around this life stage. Student-focused digital products and retirement planning services represent growth opportunities at either end of the age spectrum.

### Q2: Average Account Balance by Education Level

In [ ]:
q2 = run_query("""
SELECT education, COUNT(*) AS total_customers,
    ROUND(AVG(balance), 2) AS avg_balance,
    ROUND(MIN(balance), 2) AS min_balance,
    ROUND(MAX(balance), 2) AS max_balance,
    ROUND(STDDEV(balance), 2) AS std_balance
FROM bank GROUP BY education ORDER BY avg_balance DESC;
""")
display(q2)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=q2, x='education', y='avg_balance', ax=ax, palette='Greens_r')
ax.set_title('Average Balance by Education Level')
ax.set_xlabel('Education')
ax.set_ylabel('Average Balance (EUR)')
plt.tight_layout()
plt.show()

**Business commentary:**  
Tertiary-educated customers hold the highest average balances at €1,846 — 42% more than secondary-educated customers (€1,296). However, the single highest balance in the dataset (€66,653) belongs to a primary-educated customer, and primary-educated customers have the widest range overall. This suggests that while education correlates with average wealth, high-net-worth individuals exist across all education levels. Wealth management and premium product targeting should not be gated purely by education — balance data is a more reliable filter.

### Q3: Loan Default Rate by Job Segment

In [ ]:
q3 = run_query("""
SELECT job, COUNT(*) AS total_customers,
    SUM(CASE WHEN \"default\" = 'yes' THEN 1 ELSE 0 END) AS defaulted,
    ROUND(100.0 * SUM(CASE WHEN \"default\" = 'yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS default_rate_pct
FROM bank GROUP BY job ORDER BY default_rate_pct DESC;
""")
display(q3)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=q3.sort_values('default_rate_pct'), x='default_rate_pct', y='job', ax=ax, palette='Reds_r')
ax.set_title('Default Rate by Job Segment (%)')
ax.set_xlabel('Default Rate (%)')
plt.tight_layout()
plt.show()

**Business commentary:**  
Entrepreneurs carry the highest default risk at 3.05%, followed by housemaids (2.92%) and the unemployed (2.24%). Students are the safest segment at just 0.28%, and retired customers also default rarely (0.64%) despite their age. This has direct implications for credit policy: loan applications from entrepreneurs and housemaids should trigger enhanced affordability assessments. Conversely, students and retirees are low-risk lending targets — an underserved opportunity for the bank.

---
## Section 2 — Product Analytics

### Q4: Housing Loan vs Personal Loan Penetration

In [ ]:
q4 = run_query("""
SELECT
    ROUND(100.0 * SUM(CASE WHEN housing = 'yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_housing_loan,
    ROUND(100.0 * SUM(CASE WHEN loan = 'yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_personal_loan,
    ROUND(100.0 * SUM(CASE WHEN housing = 'yes' AND loan = 'yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_both_loans,
    ROUND(100.0 * SUM(CASE WHEN housing = 'no' AND loan = 'no' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_no_loans
FROM bank;
""")
display(q4)

labels = ['Housing Loan Only', 'Personal Loan Only', 'Both Loans', 'No Loans']
sizes = [
    float(q4['pct_housing_loan'].iloc[0]) - float(q4['pct_both_loans'].iloc[0]),
    float(q4['pct_personal_loan'].iloc[0]) - float(q4['pct_both_loans'].iloc[0]),
    float(q4['pct_both_loans'].iloc[0]),
    float(q4['pct_no_loans'].iloc[0])
]
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90)
ax.set_title('Loan Product Penetration')
plt.tight_layout()
plt.show()

**Business commentary:**  
47.3% of customers hold a housing loan, but only 13.1% have a personal loan. Notably, 47.1% of customers have no loans at all — nearly half the customer base. This represents a significant cross-sell opportunity. Customers with a housing loan but no personal loan (roughly 40% of the base) are a warm target for personal loan campaigns, as they have already demonstrated creditworthiness and a relationship with the bank.

### Q5: Term Deposit Conversion by Customer Segment

In [ ]:
q5 = run_query("""
SELECT job, education, COUNT(*) AS total_customers,
    SUM(CASE WHEN deposit = 'yes' THEN 1 ELSE 0 END) AS converted,
    ROUND(100.0 * SUM(CASE WHEN deposit = 'yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS conversion_rate_pct
FROM bank GROUP BY job, education HAVING COUNT(*) >= 50
ORDER BY conversion_rate_pct DESC LIMIT 10;
""")
display(q5)

fig, ax = plt.subplots(figsize=(10, 5))
q5['segment'] = q5['job'] + ' / ' + q5['education']
sns.barplot(data=q5, x='conversion_rate_pct', y='segment', ax=ax, palette='Blues_r')
ax.set_title('Top 10 Segments by Term Deposit Conversion Rate')
ax.set_xlabel('Conversion Rate (%)')
plt.tight_layout()
plt.show()

**Business commentary:**  
Students with secondary education convert at 79.9% — the highest of any segment. Retired customers with tertiary education (72.1%) and students with tertiary education (71.1%) follow closely. The pattern is clear: students and retirees are the bank's most receptive audiences for term deposits. This makes intuitive sense — students are building savings habits, and retirees are seeking safe, low-risk returns. Campaign resources should be disproportionately directed at these two groups.

### Q6: Conversion Rate by Contact Method

In [ ]:
q6 = run_query("""
SELECT contact, COUNT(*) AS total_contacts,
    SUM(CASE WHEN deposit = 'yes' THEN 1 ELSE 0 END) AS conversions,
    ROUND(100.0 * SUM(CASE WHEN deposit = 'yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS conversion_rate_pct
FROM bank WHERE contact IS NOT NULL
GROUP BY contact ORDER BY conversion_rate_pct DESC;
""")
display(q6)

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=q6, x='contact', y='conversion_rate_pct', ax=ax, palette='Oranges_r')
ax.set_title('Conversion Rate by Contact Method')
ax.set_ylabel('Conversion Rate (%)')
plt.tight_layout()
plt.show()

**Business commentary:**  
Cellular contact (54.3%) outperforms telephone (50.4%), but the most striking finding is that customers with unknown contact type convert at only 22.6% — less than half the rate of cellular. This means that simply having a mobile number on file for a customer nearly doubles the likelihood of a successful campaign outcome. Data quality and contact capture should be treated as a frontline commercial priority, not just a CRM hygiene task.

---
## Section 3 — Campaign Performance

### Q7: Contacts per Campaign and Success Rate

In [ ]:
q7 = run_query("""
SELECT campaign AS contacts_this_campaign, COUNT(*) AS total_customers,
    SUM(CASE WHEN deposit = 'yes' THEN 1 ELSE 0 END) AS conversions,
    ROUND(100.0 * SUM(CASE WHEN deposit = 'yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS conversion_rate_pct
FROM bank GROUP BY campaign ORDER BY campaign;
""")
display(q7)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(q7['contacts_this_campaign'], q7['conversion_rate_pct'], marker='o', color='steelblue')
ax.axvline(x=3, color='red', linestyle='--', label='Recommended cutoff (3 contacts)')
ax.set_title('Conversion Rate by Number of Campaign Contacts')
ax.set_xlabel('Number of Contacts in This Campaign')
ax.set_ylabel('Conversion Rate (%)')
ax.legend()
plt.tight_layout()
plt.show()

**Business commentary:**  
The first contact yields the highest conversion rate at 53.4%. By the 4th contact this has dropped to 41.1%, and by the 8th contact it falls to 25%. The marginal return on each additional contact is sharply negative. The bank should implement a hard policy of no more than 3 contacts per customer per campaign cycle — beyond this point, the cost of the contact exceeds the incremental conversion value, and persistent outreach risks damaging the customer relationship.

### Q8: Campaign Conversion Rate by Month

In [ ]:
q8 = run_query("""
SELECT month, COUNT(*) AS total_contacts,
    SUM(CASE WHEN deposit = 'yes' THEN 1 ELSE 0 END) AS conversions,
    ROUND(100.0 * SUM(CASE WHEN deposit = 'yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS conversion_rate_pct
FROM bank GROUP BY month ORDER BY conversion_rate_pct DESC;
""")
display(q8)

month_order = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
q8['month_num'] = q8['month'].map({m: i for i, m in enumerate(month_order)})
q8 = q8.sort_values('month_num')

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()
ax1.bar(q8['month'], q8['total_contacts'], alpha=0.4, color='steelblue', label='Total Contacts')
ax2.plot(q8['month'], q8['conversion_rate_pct'], marker='o', color='red', label='Conversion Rate %')
ax1.set_ylabel('Total Contacts')
ax2.set_ylabel('Conversion Rate (%)')
ax1.set_title('Campaign Volume vs Conversion Rate by Month')
fig.legend(loc='upper right', bbox_to_anchor=(1,1), bbox_transform=ax1.transAxes)
plt.tight_layout()
plt.show()

**Business commentary:**  
December (90.9%), March (89.9%), September (84.3%) and October (82.4%) are the best months for term deposit campaigns. May is the single worst month at 32.8% — yet it accounts for 2,824 contacts, the highest volume of any month. The bank is deploying its largest campaign effort in its least effective window. Reallocating even 30% of May contacts to September or October would materially improve overall campaign ROI without increasing total spend.

### Q9: Previous Contacts vs Conversion Rate

In [ ]:
q9 = run_query("""
SELECT
    CASE WHEN previous = 0 THEN '0 - Never contacted'
         WHEN previous BETWEEN 1 AND 2 THEN '1-2 prior contacts'
         WHEN previous BETWEEN 3 AND 5 THEN '3-5 prior contacts'
         ELSE '6+ prior contacts' END AS prior_contact_band,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN deposit = 'yes' THEN 1 ELSE 0 END) AS conversions,
    ROUND(100.0 * SUM(CASE WHEN deposit = 'yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS conversion_rate_pct
FROM bank GROUP BY prior_contact_band ORDER BY conversion_rate_pct DESC;
""")
display(q9)

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=q9, x='prior_contact_band', y='conversion_rate_pct', ax=ax, palette='Purples_r')
ax.set_title('Conversion Rate by Prior Campaign Contact History')
ax.set_ylabel('Conversion Rate (%)')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

**Business commentary:**  
Customers contacted 3–5 times in previous campaigns convert at 69.4%, compared to just 40.7% for those never previously contacted. Re-engaging warm leads is 70% more effective than cold outreach. This strongly supports building a structured re-engagement programme: customers who were contacted in prior campaigns but did not convert should be prioritised in future campaign lists over net-new cold contacts.

---
## Section 4 — Risk Analysis

### Q10: Default Rate by Age Group and Job Type

In [ ]:
q10 = run_query("""
SELECT
    CASE WHEN age < 25 THEN 'Under 25' WHEN age BETWEEN 25 AND 34 THEN '25-34'
         WHEN age BETWEEN 35 AND 44 THEN '35-44' WHEN age BETWEEN 45 AND 54 THEN '45-54'
         WHEN age BETWEEN 55 AND 64 THEN '55-64' ELSE '65+' END AS age_group,
    job, COUNT(*) AS total_customers,
    SUM(CASE WHEN \"default\" = 'yes' THEN 1 ELSE 0 END) AS defaulted,
    ROUND(100.0 * SUM(CASE WHEN \"default\" = 'yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS default_rate_pct
FROM bank GROUP BY age_group, job HAVING COUNT(*) >= 20
ORDER BY default_rate_pct DESC LIMIT 10;
""")
display(q10)

**Business commentary:**  
Young housemaids aged 25–34 have the highest default rate of any age-job combination at 8.57% — more than five times the overall rate of 1.62%. Self-employed customers aged 35–44 (3.97%) and middle-aged entrepreneurs (3.1–3.5%) are also significantly elevated. These combinations should trigger enhanced due diligence in credit assessments. Notably, the risk is not just about job type — age compounds it. A 45-year-old entrepreneur is materially riskier than a 30-year-old one.

### Q11: High Balance + High Default Risk Segments

In [ ]:
q11 = run_query("""
WITH segment_stats AS (
    SELECT job, education, COUNT(*) AS total_customers,
        ROUND(AVG(balance), 2) AS avg_balance,
        ROUND(100.0 * SUM(CASE WHEN \"default\" = 'yes' THEN 1 ELSE 0 END) / COUNT(*), 2) AS default_rate_pct
    FROM bank GROUP BY job, education HAVING COUNT(*) >= 30
)
SELECT * FROM segment_stats
WHERE avg_balance > (SELECT AVG(balance) FROM bank)
  AND default_rate_pct > (SELECT 100.0 * SUM(CASE WHEN \"default\" = 'yes' THEN 1 ELSE 0 END) / COUNT(*) FROM bank)
ORDER BY default_rate_pct DESC;
""")
display(q11)

fig, ax = plt.subplots(figsize=(8, 5))
q11['segment'] = q11['job'] + ' / ' + q11['education']
scatter = ax.scatter(q11['avg_balance'], q11['default_rate_pct'], s=q11['total_customers']*2, alpha=0.7, color='crimson')
for _, row in q11.iterrows():
    ax.annotate(row['segment'], (row['avg_balance'], row['default_rate_pct']), fontsize=8, ha='left')
ax.set_title('High Balance + High Default Risk Segments')
ax.set_xlabel('Average Balance (EUR)')
ax.set_ylabel('Default Rate (%)')
plt.tight_layout()
plt.show()

**Business commentary:**  
Six segments hold above-average balances yet default at above-average rates — the paradox segments. The most concerning are unemployed primary-educated customers (default rate 7.14%, avg balance €1,607) and tertiary-educated entrepreneurs (6.06%, avg balance €2,090). A high account balance does not guarantee creditworthiness. These customers may be drawing down savings rather than earning income. The bank should implement early warning indicators — such as declining balance trends — to trigger proactive outreach before these customers enter default.

---
## Section 5 — Advanced Window Functions

### Q12: Customer Balance Rank Within Job Segment

In [ ]:
q12 = run_query("""
SELECT job, age, balance, education,
    RANK() OVER (PARTITION BY job ORDER BY balance DESC) AS balance_rank_in_job,
    ROUND(PERCENT_RANK() OVER (PARTITION BY job ORDER BY balance) * 100, 1) AS balance_percentile
FROM bank ORDER BY job, balance_rank_in_job LIMIT 50;
""")
display(q12)

**Business commentary:**  
Ranking customers by balance within their own job segment allows the bank to identify the top tier within each category — not just globally. A management customer in the top 10% of their peer group may be a better private banking prospect than a retired customer with the same absolute balance, because relative wealth within a segment signals stronger financial management behaviour. This ranking approach supports relationship manager prioritisation and tiered service allocation.

### Q13: Running Total of Conversions by Month

In [ ]:
q13 = run_query("""
WITH monthly_conversions AS (
    SELECT month, SUM(CASE WHEN deposit = 'yes' THEN 1 ELSE 0 END) AS monthly_conversions
    FROM bank GROUP BY month
)
SELECT month, monthly_conversions,
    SUM(monthly_conversions) OVER (
        ORDER BY CASE month
            WHEN 'jan' THEN 1 WHEN 'feb' THEN 2 WHEN 'mar' THEN 3 WHEN 'apr' THEN 4
            WHEN 'may' THEN 5 WHEN 'jun' THEN 6 WHEN 'jul' THEN 7 WHEN 'aug' THEN 8
            WHEN 'sep' THEN 9 WHEN 'oct' THEN 10 WHEN 'nov' THEN 11 WHEN 'dec' THEN 12
        END
    ) AS running_total_conversions
FROM monthly_conversions
ORDER BY CASE month
    WHEN 'jan' THEN 1 WHEN 'feb' THEN 2 WHEN 'mar' THEN 3 WHEN 'apr' THEN 4
    WHEN 'may' THEN 5 WHEN 'jun' THEN 6 WHEN 'jul' THEN 7 WHEN 'aug' THEN 8
    WHEN 'sep' THEN 9 WHEN 'oct' THEN 10 WHEN 'nov' THEN 11 WHEN 'dec' THEN 12
END;
""")
display(q13)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(q13['month'], q13['monthly_conversions'], label='Monthly Conversions', alpha=0.6, color='steelblue')
ax.plot(q13['month'], q13['running_total_conversions'], color='red', marker='o', label='Running Total')
ax.set_title('Monthly vs Running Total Term Deposit Conversions')
ax.legend()
plt.tight_layout()
plt.show()

**Business commentary:**  
The running total chart shows that conversion accumulation is front-loaded in May due to sheer volume, but the rate of accumulation slows significantly in summer months. The steepest cumulative gains come from March and the Sep–Dec window where both volume and conversion rate are favourable. A campaign manager tracking against an annual conversion target would want to weight effort heavily toward Q1 and Q4, rather than the traditional mid-year push.

### Q14: Top 3 Balance Customers Per Job Category

In [ ]:
q14 = run_query("""
WITH ranked AS (
    SELECT job, age, balance, education, marital,
        ROW_NUMBER() OVER (PARTITION BY job ORDER BY balance DESC) AS rn
    FROM bank
)
SELECT job, age, balance, education, marital, rn AS rank_within_job
FROM ranked WHERE rn <= 3 ORDER BY job, rn;
""")
display(q14)

**Business commentary:**  
The top 3 customers per job segment form the basis of a priority outreach list for premium product offers — private banking, structured deposits, or wealth management services. Note that the top balance in the primary-educated segment (€66,653) exceeds the top balance in the management segment, reinforcing that job title is a poor standalone proxy for wealth. Any premium product eligibility framework should be balance-driven, not job-driven.

### Q15: Month-over-Month Change in Average Balance

In [ ]:
q15 = run_query("""
WITH monthly_avg AS (
    SELECT month,
        CASE month WHEN 'jan' THEN 1 WHEN 'feb' THEN 2 WHEN 'mar' THEN 3 WHEN 'apr' THEN 4
            WHEN 'may' THEN 5 WHEN 'jun' THEN 6 WHEN 'jul' THEN 7 WHEN 'aug' THEN 8
            WHEN 'sep' THEN 9 WHEN 'oct' THEN 10 WHEN 'nov' THEN 11 WHEN 'dec' THEN 12
        END AS month_num,
        ROUND(AVG(balance), 2) AS avg_balance
    FROM bank GROUP BY month
)
SELECT month, avg_balance,
    LAG(avg_balance) OVER (ORDER BY month_num) AS prev_month_avg,
    ROUND(avg_balance - LAG(avg_balance) OVER (ORDER BY month_num), 2) AS mom_change,
    ROUND(100.0 * (avg_balance - LAG(avg_balance) OVER (ORDER BY month_num))
          / NULLIF(LAG(avg_balance) OVER (ORDER BY month_num), 0), 2) AS mom_change_pct
FROM monthly_avg ORDER BY month_num;
""")
display(q15)

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['green' if x >= 0 else 'red' for x in q15['mom_change'].fillna(0)]
ax.bar(q15['month'], q15['mom_change'].fillna(0), color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Month-over-Month Change in Average Customer Balance (EUR)')
ax.set_ylabel('Change in EUR')
plt.tight_layout()
plt.show()

**Business commentary:**  
Average balances are highly volatile — swinging by up to ±49% month-on-month. The sharpest drops are in May (−31.3%) and July (−33.7%), which coincide with the bank's heaviest campaign months. This could indicate that customers being heavily targeted are financially stressed or withdrawing funds. The strongest recovery occurs in Q4 (Oct–Dec), with balances climbing steadily to a peak of €2,735 in December — likely reflecting year-end bonuses and savings behaviour. Deposit campaigns in Q4 can capitalise on this natural liquidity cycle.

---
## Summary Findings

| Theme | Key Finding | Business Recommendation |
|---|---|---|
| Customer Profiling | Tertiary-educated customers hold 42% higher balances; students & retirees are lowest risk | Target wealth products by balance, not job title |
| Product Analytics | 47% of customers have no loans; students & retirees convert at 70%+ for term deposits | Cross-sell loans to housing loan holders; focus deposit campaigns on students & retirees |
| Campaign Performance | May is the worst conversion month yet highest volume; 1st contact converts at 53% vs 25% at 8th | Shift budget to Sep/Oct/Mar; cap outreach at 3 contacts |
| Risk Analysis | Entrepreneurs (3.05%) and housemaids (2.92%) carry the highest default risk; high-balance segments can still default at 6-7% | Implement early warning triggers for paradox segments; tighten credit criteria for entrepreneurs |